# Evaluation of the MAD paper

# Is the toxic agent winning?

In [2]:
import pandas as pd
import json
from pathlib import Path
from scipy import stats

In [3]:
df = pd.read_csv("../data/evaluation mad/convergence_of_480_toxic_random_discussions_topic_16.csv", sep =",")

In [4]:
print(df["reason_for_convergence"].agg("value_counts"))
print(df["which_agent_is_toxic"].agg("value_counts"))
df["toxicity_level"].agg("value_counts")

df.groupby("toxicity_level")

reason_for_convergence
moderator detected alignment    189
pro has been convinced          152
con has been convinced          139
Name: count, dtype: int64
which_agent_is_toxic
pro has toxic behaviour    247
con has toxic behaviour    233
Name: count, dtype: int64


In [5]:
dt = df[df["reason_for_convergence"] != "moderator detected alignment"]

In [6]:
dt.groupby("toxicity_level")["reason_for_convergence"].value_counts(normalize=True)

toxicity_level  reason_for_convergence
mild            pro has been convinced    0.505376
                con has been convinced    0.494624
moderate        pro has been convinced    0.538462
                con has been convinced    0.461538
no              pro has been convinced    0.521277
                con has been convinced    0.478723
Name: proportion, dtype: float64

In [7]:
test_anova = dt.groupby("which_agent_is_toxic")["reason_for_convergence"].value_counts()

In [8]:
"""f_stats, p_value = stats.f_oneway(*[df[col] for col in test_anova])
print(f"ANOVA F-statistic: {f_stats}, p-value: {p_value}")"""

'f_stats, p_value = stats.f_oneway(*[df[col] for col in test_anova])\nprint(f"ANOVA F-statistic: {f_stats}, p-value: {p_value}")'

In [9]:
dt.groupby("toxicity_level")["rounds_to_convergence"].agg(["mean", "var", "size"])

,mean,var,size
toxicity_level,,,
mild,11.376344,8.628565,93
moderate,11.240385,7.873693,104
no,9.531915,7.735530,94


### T-Test for the toxic Agent winning

In [191]:
con_outcomes = (dt[dt["which_agent_is_toxic"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_outcomes, popmean=0.5)
print(f"win rate: {con_outcomes.mean():.4f}, p={p_value:.4f}")

win rate: nan, p=nan


/var/folders/v6/9c6hcq397n90s3n8znn3v7vh0000gn/T/ipykernel_67973/370564180.py:2: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t_stat, p_value = stats.ttest_1samp(con_outcomes, popmean=0.5)


# Is the Agent winning that starts the discussion?

In [67]:
directory = Path('../data/toxic_and_baseline_random_mad1')
file_lookup = {f.name: f for f in directory.rglob("*.json")}

## Extract the Agent with the opening Statement

In [72]:
def get_agent_with_opening_statement(path):
    full_path = file_lookup[Path(path).name]
    with open(full_path) as fp:
        data = json.load(fp)

    opening_agent = data["opening_statements"][0]["agent"].split("_")[0]  # "pro" or "con"

    return opening_agent

In [73]:
dt["opening_agent"] = dt["Path"].apply(get_agent_with_opening_statement)

In [87]:
dt.groupby("opening_agent")["reason_for_convergence"].value_counts(normalize=True)

opening_agent  reason_for_convergence
con            con has been convinced    0.507463
               pro has been convinced    0.492537
pro            pro has been convinced    0.547771
               con has been convinced    0.452229
Name: proportion, dtype: float64

### t-test for the opener of the discussion

In [178]:
con_winning = (dt[dt["opening_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_loosing = (dt[dt["opening_agent"] == "con"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_winning, con_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: -0.2426
p-value: 0.8085


In [179]:
pro_winning = (dt[dt["opening_agent"] == "pro"]["reason_for_convergence"] == "pro has been convinced").astype(int)
pro_loosing = (dt[dt["opening_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_winning, pro_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 1.6954
p-value: 0.0910


## Extract the Agent who starts the Discussion

In [75]:
def get_agent_who_starts(path):
    full_path = file_lookup[Path(path).name]
    with open(full_path, 'r') as fp:
        data = json.load(fp)

    nrounds = data["nrounds"]
    starting_agent = data["discussion_history"][f"Round_{nrounds}"][0]["agent"].split("_")[0]  # "pro" or "con"

    return starting_agent

In [78]:
dt["starting_agent"] = dt["Path"].apply(get_agent_who_starts)

In [79]:
dt.groupby("starting_agent")["reason_for_convergence"].value_counts(normalize=True)

starting_agent  reason_for_convergence
con             pro has been convinced    0.718519
                con has been convinced    0.281481
pro             con has been convinced    0.647436
                pro has been convinced    0.352564
Name: proportion, dtype: float64

### T-Test for the starter of the discussion
Hypothesis: The agent that starts the discussion is more likely to win the discussion.



In [183]:
con_winning = (dt[dt["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_loosing = (dt[dt["starting_agent"] == "con"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_winning, con_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 7.9120
p-value: 0.0000


In [184]:
pro_winning = (dt[dt["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
pro_loosing = (dt[dt["starting_agent"] == "pro"]["reason_for_convergence"] == "pro has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_winning, pro_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 5.4333
p-value: 0.0000


### One-Sample T-Test

In [185]:
# when con starts, do they win more than 50%?
con_outcomes = (dt[dt["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_outcomes, popmean=0.5)
print(f"CON starter:")
print(f"  win rate: {con_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

# when pro starts, do they win more than 50%?
pro_outcomes = (dt[dt["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(pro_outcomes, popmean=0.5)
print(f"PRO starter:")
print(f"  win rate: {pro_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

CON starter:
  win rate: 0.7132
  t-statistic: 5.4783
  p-value: 0.0000
PRO starter:
  win rate: 0.6474
  t-statistic: 3.8419
  p-value: 0.0002


### Anova for the starter of the discussion

In [187]:
dt["group"] = dt["starting_agent"] + " | " + dt["reason_for_convergence"]

groups = [group["starter_wins"].values for _, group in dt.groupby("group")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# group means for overview
print(dt.groupby("group")["starter_wins"].mean())

F-statistic: 0.6619
P-value: 0.5761
group
con | con has been convinced    0.526316
con | pro has been convinced    0.567010
pro | con has been convinced    0.475248
pro | pro has been convinced    0.563636
Name: starter_wins, dtype: float64
